# Step 2: Data Checks, Cleaning, and Target Definition

**Goal:** Inspect data anomalies, handle missing values and types, prevent data leakage, and establish our success metric.

---
## 2A. Suspicious Things & Sanity Checks (No Changes Yet)
In this section, we audit:
1. `popularity == 99999` (Sentinel values)
2. `episodes == 0` and extreme episode counts
3. `rating` missingness (Low-watcher niche shows vs. newly released shows)
4. Duplicates by `url` and by `(name + country)`
5. `watchers` string formatting and integer conversion

In [1]:
import pandas as pd
import numpy as np

# Load raw scraped dataset without modifying it
df_raw = pd.read_csv('../data/raw/dramas_raw.csv')
print(f'Loaded {df_raw.shape[0]} rows and {df_raw.shape[1]} columns.')

Loaded 10400 rows and 20 columns.


### 1. Check Popularity: What is `99999`?
We investigate if `99999` represents an unranked placeholder for shows that do not have enough platform engagement.

In [2]:
pop_99999 = df_raw[df_raw['popularity'] == 99999]
pop_ranked = df_raw[df_raw['popularity'] != 99999]

print(f"Count of popularity == 99999: {len(pop_99999)} ({len(pop_99999)/len(df_raw)*100:.2f}%)")
print(f"Max ranked popularity score: {pop_ranked['popularity'].max()}")

# Clean watchers temporarily to compare distributions
temp_watchers = pd.to_numeric(df_raw['watchers'].astype(str).str.replace(',', '').str.strip(), errors='coerce')

print('\n--- Watchers for Popularity == 99999 ---')
print(temp_watchers[df_raw['popularity'] == 99999].describe())

print('\n--- Watchers for Popularity != 99999 (Ranked Titles) ---')
print(temp_watchers[df_raw['popularity'] != 99999].describe())

Count of popularity == 99999: 4338 (41.71%)
Max ranked popularity score: 24671

--- Watchers for Popularity == 99999 ---
count    4338.000000
mean       64.706547
std        55.274179
min         0.000000
25%        17.000000
50%        49.000000
75%       102.000000
max       199.000000
Name: watchers, dtype: float64

--- Watchers for Popularity != 99999 (Ranked Titles) ---
count      6062.000000
mean      10010.937149
std       24754.687634
min         200.000000
25%         520.250000
50%        1613.000000
75%        7344.750000
max      327227.000000
Name: watchers, dtype: float64


### 2. Check Episodes: Zeroes and Outliers
Are there dramas with 0 episodes? What are the minimum and maximum episode counts?

In [3]:
print(f"Min episodes: {df_raw['episodes'].min()}")
print(f"Max episodes: {df_raw['episodes'].max()}")
print(f"Count of episodes == 0: {(df_raw['episodes'] == 0).sum()} ({(df_raw['episodes'] == 0).mean()*100:.2f}%)")

print('\nSample titles with episodes == 0:')
zero_eps = df_raw[df_raw['episodes'] == 0][['name', 'country', 'aired_bw', 'url']].head(5)
print(zero_eps.to_string(index=False))

print('\nSample titles with highest episode counts:')
high_eps = df_raw.sort_values(by='episodes', ascending=False)[['name', 'episodes', 'country', 'aired_bw']].head(5)
print(high_eps.to_string(index=False))

Min episodes: 0
Max episodes: 450
Count of episodes == 0: 18 (0.17%)

Sample titles with episodes == 0:
                                           name     country                    aired_bw                                                                     url
                     Allergic to Bangkok (2016)    Thailand            Nov 30, 2016 - ?                       https://mydramalist.com/22430-allergic-to-bangkok
                       Luang Phi Digital (2015)    Thailand Mar 14, 2015 - Jan  2, 2016                         https://mydramalist.com/24120-luang-phi-digital
                       Koisuru JK Zombie (2016)       Japan                Dec 29, 2016                         https://mydramalist.com/22050-koisuru-jk-zombie
                    The Phantom Atom Boy (2016)       Japan         Jul 25, 2016 - 2016                      https://mydramalist.com/54963-the-phantom-atom-boy
True Education by My Billionaire Teacher (2025) South Korea                Jun 18, 2025 https://

### 3. Check Rating Missingness: Low Watchers vs. New Releases
Why are 1,465 ratings missing? Is it because they are brand new, or because they lack viewer votes?

In [4]:
missing_rating = df_raw[df_raw['rating'].isna()]
present_rating = df_raw[df_raw['rating'].notna()]

print(f"Total missing ratings: {len(missing_rating)} ({len(missing_rating)/len(df_raw)*100:.2f}%)")
print(f"Median watchers for missing rating: {temp_watchers[df_raw['rating'].isna()].median()}")
print(f"Median watchers for present rating: {temp_watchers[df_raw['rating'].notna()].median()}")
print(f"Missing rating titles with popularity == 99999: {(missing_rating['popularity'] == 99999).mean()*100:.2f}%")

# Extract release year to test if missingness is concentrated only in recent years
years = df_raw['aired_bw'].astype(str).str.extract(r'(\d{4})')[0].astype(float)
print('\nYear distribution for missing ratings (Top 6 years):')
print(years[df_raw['rating'].isna()].value_counts().head(6))
print('\nYear distribution for present ratings (Top 6 years):')
print(years[df_raw['rating'].notna()].value_counts().head(6))

Total missing ratings: 1465 (14.09%)
Median watchers for missing rating: 12.0
Median watchers for present rating: 548.0
Missing rating titles with popularity == 99999: 98.84%

Year distribution for missing ratings (Top 6 years):
0
2022.0    249
2021.0    241
2016.0    165
2015.0    151
2017.0    136
2023.0    126
Name: count, dtype: int64

Year distribution for present ratings (Top 6 years):
0
2021.0    1503
2022.0    1434
2017.0    1086
2016.0    1024
2015.0     880
2018.0     869
Name: count, dtype: int64


### 4. Check Duplicates: URLs vs. (Name + Country)
Check whether rows share the same URL or the same name and country.

In [5]:
dup_urls = df_raw[df_raw.duplicated(subset=['url'], keep=False)]
print(f"Duplicate URLs count: {len(dup_urls)}")

dup_name_country = df_raw[df_raw.duplicated(subset=['name', 'country'], keep=False)]
print(f"Duplicate (name + country) count: {len(dup_name_country)} ({len(dup_name_country)//2} pairs)")

# Inspect sample duplicate pairs
print('\nSample (name + country) duplicate pairs:')
sample_dups = dup_name_country.sort_values(by=['name', 'country'])[['name', 'country', 'aired_bw', 'rating', 'watchers', 'url']].head(6)
print(sample_dups.to_string(index=False))

Duplicate URLs count: 0
Duplicate (name + country) count: 26 (13 pairs)

Sample (name + country) duplicate pairs:
                                           name country                    aired_bw  rating watchers                                                                     url
Every Moment That I Think of You Special (2021)   China Aug 16, 2021 - Aug 17, 2021     6.8      372 https://mydramalist.com/720195-every-moment-that-i-think-of-you-special
Every Moment That I Think of You Special (2021)   China                Aug 15, 2021     6.7      222 https://mydramalist.com/720199-every-moment-that-i-think-of-you-special
                            Fall in Love (2021)   China Nov  9, 2021 - Dec  8, 2021     8.3   24,136                              https://mydramalist.com/53549-fall-in-love
                            Fall in Love (2021)   China Feb  8, 2021 - Mar  2, 2021     7.3      929              https://mydramalist.com/688541-chong-sheng-zhi-wei-ying-di
                   Fa

### 5. Check Watchers: String with Commas to Numbers
Verify that every watcher value can be converted safely from strings with commas into standard 64-bit integers.

In [6]:
# Test conversion
cleaned_watchers_series = df_raw['watchers'].astype(str).str.replace(',', '').str.strip()
non_numeric_count = (~cleaned_watchers_series.str.isnumeric()).sum()
print(f"Non-numeric rows in watchers: {non_numeric_count}")

converted_watchers = cleaned_watchers_series.astype('int64')
print(f"Successfully converted all {len(converted_watchers)} rows to int64.")
print(f"Min watchers: {converted_watchers.min()} | Max watchers: {converted_watchers.max():,} | Mean: {converted_watchers.mean():,.1f}")

Non-numeric rows in watchers: 0
Successfully converted all 10400 rows to int64.
Min watchers: 0 | Max watchers: 327,227 | Mean: 5,862.2


---
## 2B. Parse Dates & Evaluate Temporal Distribution

**Goals:**
1. Parse `aired_bw` into `start_date` and `end_date` (datetime objects).
2. Extract `release_year` (with fallback to title year `(YYYY)` for missing dates).
3. Audit parse failure causes.
4. Examine temporal distribution across years and countries to prepare for our **time-based train/test split**.

In [7]:
def parse_aired_dates(val):
    """
    Parses aired_bw string into (start_date, end_date).
    Handles single dates, ranges, and open-ended dates ('?').
    """
    if pd.isna(val):
        return None, None
    parts = str(val).split(' - ')
    start_s = parts[0].strip()
    end_s = parts[1].strip() if len(parts) > 1 else start_s
    
    # Convert using mixed format parsing
    start_dt = pd.to_datetime(start_s, format='mixed', errors='coerce')
    end_dt = pd.to_datetime(end_s, format='mixed', errors='coerce') if end_s != '?' else None
    return start_dt, end_dt

# Apply parsing
parsed_dates = df_raw['aired_bw'].apply(parse_aired_dates)
df_raw['start_date'] = [p[0] for p in parsed_dates]
df_raw['end_date'] = [p[1] for p in parsed_dates]

# Extract release_year (Primary: start_date year; Fallback: year in title)
df_raw['release_year'] = df_raw['start_date'].dt.year
title_year = df_raw['name'].str.extract(r'\((\d{4})\)')[0].astype(float)
df_raw['release_year'] = df_raw['release_year'].fillna(title_year).astype('Int64')

print(f"start_date nulls: {df_raw['start_date'].isna().sum()} (matches 77 raw NaN rows)")
print(f"end_date nulls: {df_raw['end_date'].isna().sum()} (77 raw NaNs + 61 open-ended '?')")
print(f"release_year nulls after title fallback: {df_raw['release_year'].isna().sum()} (100% complete!)")

start_date nulls: 77 (matches 77 raw NaN rows)
end_date nulls: 138 (77 raw NaNs + 61 open-ended '?')
release_year nulls after title fallback: 0 (100% complete!)


### Parse Failure Breakdown
Let's verify why rows failed date parsing:
1. **77 rows:** `aired_bw` was originally missing (`NaN`). All 77 had their year recovered from `name` (e.g. `(2018)`).
2. **61 rows:** `start_date` parsed cleanly, but `end_date` failed because the value was `?` (broadcast end unannounced/open-ended).

In [8]:
# Sample of the 61 open-ended dates
open_ended = df_raw[df_raw['start_date'].notna() & df_raw['end_date'].isna()][['name', 'aired_bw', 'country', 'episodes']]
print("Sample of open-ended dates (' - ?'):")
print(open_ended.head(5).to_string(index=False))

# Sample of the 77 raw NaNs recovered from title
recovered_nans = df_raw[df_raw['start_date'].isna()][['name', 'release_year', 'country']]
print("\nSample of recovered NaN release years from title:")
print(recovered_nans.head(5).to_string(index=False))

Sample of open-ended dates (' - ?'):
                                name         aired_bw  country  episodes
Ishiki Takasugi! Takasugi-kun (2018) Jan 29, 2018 - ?    Japan        38
        Test Content Season 4 (2018) Mar  6, 2018 - ?    Japan        32
          Allergic to Bangkok (2016) Nov 30, 2016 - ? Thailand         0
                     Rub Code (2016) Dec 23, 2016 - ?    China        12
            Mu Guang Zhi Tong (2016) Sep  7, 2016 - ?    China        17

Sample of recovered NaN release years from title:
                               name  release_year country
           My Roommate Diary (2018)          2018   China
         Running Out of Time (2018)          2018   China
             Chosen Season 3 (2018)          2018   China
              Heart Planning (2018)          2018   China
Love and Lose Heart Season 2 (2018)          2018   China


### Temporal Distribution Across Years and Countries
We compute a crosstabulation of dramas by `release_year` and `country` to evaluate data density over time.

In [9]:
print("--- Overall Drama Count Per Year ---")
print(df_raw['release_year'].value_counts().sort_index().to_string())

print("\n--- Crosstab: Drama Count by Release Year & Country ---")
year_country_ct = pd.crosstab(df_raw['release_year'], df_raw['country'], margins=True, margins_name='Total')
print(year_country_ct.to_string())

--- Overall Drama Count Per Year ---
release_year
2015    1048
2016    1198
2017    1235
2018     976
2019     559
2020     184
2021    1770
2022    1684
2023     801
2024     569
2025     376

--- Crosstab: Drama Count by Release Year & Country ---
country       China  Japan  South Korea  Thailand  Total
release_year                                            
2015            280    390          186       192   1048
2016            329    402          212       255   1198
2017            333    412          260       230   1235
2018            344     96          296       240    976
2019             14      0          315       230    559
2020              0      0           31       153    184
2021            724    485          331       230   1770
2022            576    526          312       270   1684
2023              0    289          234       278    801
2024              0      0          279       290    569
2025              0      0          144       232    376
Total    